# REMS training table: sol profile, hold-out rule, `gale_rems_training.csv`

Handover brief steps 1 and 2. One row per REMS 15-minute bin, with rover-pixel covariates and a leakage-safe split.

**Hold-out rule (decided up front).** Sols are grouped into contiguous campaign blocks (a new block starts after a gap of more than 50 sols). Whole blocks are held out, never random rows and never single sols. Two blocks are sealed as a final test set; the rest are used for leave-one-block-out cross-validation. After validation the final model is refit on everything.

**Dust opacity.** `tau` comes from Mastcam direct-Sun images (Lemmon 2023): the mean of the 440 and 880 nm optical depths per sol, linearly interpolated across gaps of up to 20 sols. Sols with no usable measurement keep `tau = 0.5` and `tau_is_placeholder = True`. The flux columns use this tau.

Run this notebook in its own kernel. It needs `numpy`, `pandas` and (last cell only) `scikit-learn`. Besides the two CSVs in `processed_data`, it reads the two Mastcam tau text files listed in `TAU_FILES`.

In [43]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path("processed_data")
REMS_CSV = DATA_DIR / "rems_binned_all_sols.csv"
FEATURES_CSV = DATA_DIR / "gale_dust_devil_features_100m.csv"
OUT_DIR = DATA_DIR

# 100 m THEMIS grid, verified against gale_dust_devil_meta.json (handover brief, Oct 24)
GRID_ROWS, GRID_COLS = 1436, 1477
T_A, T_C = 0.001687051876845213, 136.7490510333193       # lon = T_C + T_A * col
T_E, T_F = -0.0016870993139129456, -4.148577212911933    # lat = T_F + T_E * row

BLOCK_GAP_SOLS = 50                                  # gap (sols) that starts a new block; 15 split same-pixel campaigns (sols 765/799, 1750/1770)
TEST_SOL_RANGES = [(1343, 1360), (3069, 3110)]       # sealed test blocks; each range must equal one whole block
MIN_BINS_CORE = 40                                   # sols below this are kept but flagged core_sol = False
TAU_PLACEHOLDER = 0.5                                # used only where no Mastcam tau is available (flagged tau_is_placeholder)

# Mastcam aerosol optical depth, Lemmon (2023), Mendeley Data doi 10.17632/tbh3pg39xc (version 3); put both files in DATA_DIR
TAU_FILES = {440: "raw_data/msl-tau-440-arch-3953.txt", 880: "raw_data/msl-tau-880-arch-3953.txt"}
TAU_MAX_GAP_SOLS = 20                                # interpolate tau only between measured sols at most this far apart

In [44]:
rems = pd.read_csv(REMS_CSV)
rems = rems.drop_duplicates()
assert not rems.duplicated(["sol", "bin"]).any()
assert (rems.groupby("sol")[["latitude_deg", "longitude_deg_e"]].nunique() == 1).all().all()   # one position per sol
assert (rems.groupby("sol").loc_row_this_sol.nunique() == 1).all()                            # flag is constant per sol
print(rems.shape, rems.sol.nunique(), "sols,", "sol", rems.sol.min(), "to", rems.sol.max())

(15523, 19) 319 sols, sol 1 to 3778


## Step 1: sol profile, blocks, split

In [45]:
def ls_deg(sol):
    """Areocentric solar longitude (deg) from sol number, Kepler orbit with Ls = 150.7 at sol 0."""
    e, period, ls_peri = 0.0934, 668.5991, 251.0
    nu0 = np.deg2rad(150.7 - ls_peri)
    ea0 = 2 * np.arctan2(np.sqrt(1 - e) * np.sin(nu0 / 2), np.sqrt(1 + e) * np.cos(nu0 / 2))
    m = ea0 - e * np.sin(ea0) + 2 * np.pi * np.asarray(sol, float) / period
    ea = m.copy()
    for _ in range(50):
        ea = ea - (ea - e * np.sin(ea) - m) / (1 - e * np.cos(ea))
    nu = 2 * np.arctan2(np.sqrt(1 + e) * np.sin(ea / 2), np.sqrt(1 - e) * np.cos(ea / 2))
    return (np.rad2deg(nu) + ls_peri) % 360

# known values from the handover brief (sol 0: 150.7, sol 800: 228.18, sol 1163: 68.1)
assert abs(ls_deg(0) - 150.7) < 0.05 and abs(ls_deg(800) - 228.2) < 0.1 and abs(ls_deg(1163) - 68.1) < 0.1

rems["hour"] = (rems.bin // 4).astype(int)
sols = rems.groupby("sol").agg(
    n_bins=("bin", "size"), hours_covered=("hour", "nunique"),
    n_ground=("ground_temp_k", "count"), n_air=("air_temp_k", "count"),
    lat=("latitude_deg", "first"), lon=("longitude_deg_e", "first"), elev_rems_m=("elevation_m", "first"),
    loc_this_sol=("loc_row_this_sol", "first"),
).reset_index().sort_values("sol").reset_index(drop=True)
sols["ls_deg"] = ls_deg(sols.sol)
sols["core_sol"] = sols.n_bins >= MIN_BINS_CORE
sols["block"] = (sols.sol.diff() > BLOCK_GAP_SOLS).cumsum()

sols["split_role"] = "train"
for lo, hi in TEST_SOL_RANGES:
    in_range = sols.sol.between(lo, hi)
    assert sols.loc[in_range, "block"].nunique() == 1, f"sols {lo}-{hi} span more than one block"
    b = sols.loc[in_range, "block"].iloc[0]
    assert (sols.block == b).sum() == in_range.sum(), f"sols {lo}-{hi} are not a whole block"
    sols.loc[sols.block == b, "split_role"] = "test"

blocks = sols.groupby("block").agg(
    sol_first=("sol", "min"), sol_last=("sol", "max"), n_sols=("sol", "size"), n_bins=("n_bins", "sum"),
    ls_first=("ls_deg", "first"), ls_last=("ls_deg", "last"),
    elev_min_m=("elev_rems_m", "min"), elev_max_m=("elev_rems_m", "max"), role=("split_role", "first"),
)
print(blocks.round(1).to_string())
print()
print(sols.groupby("split_role").agg(sols=("sol", "size"), bins=("n_bins", "sum")))

# independence check: how close is each test block to the training data, in sols and in km on the ground
train = sols[sols.split_role == "train"]
for b, g in sols[sols.split_role == "test"].groupby("block"):
    d_sols = np.abs(train.sol.to_numpy()[:, None] - g.sol.to_numpy()[None, :]).min()
    d_km = np.hypot((g.lat.mean() - train.lat) * 59.16, (g.lon.mean() - train.lon) * 59.0).min()
    print(f"test block {b} (sols {g.sol.min()}-{g.sol.max()}): nearest training sol {d_sols} sols away, nearest training position {d_km:.2f} km away")

       sol_first  sol_last  n_sols  n_bins  ls_first  ls_last  elev_min_m  elev_max_m   role
block                                                                                       
0              1       170      90    4040     151.2    253.3     -4521.0     -4501.0  train
1            365       375      11     597       7.7     12.7     -4503.4     -4501.7  train
2            584       865      53    2562     108.9    270.4     -4479.6     -4452.8  train
3           1160      1165       6     249      66.8     69.0     -4433.9     -4432.3  train
4           1343      1360      12     561     153.8    163.0     -4435.6     -4429.2   test
5           1650      1865      40    2013     340.5     83.1     -4299.9     -4174.1  train
6           2550      2673      24    1161      90.5    150.0     -4133.4     -4095.4  train
7           2871      2968      23    1261     270.6    329.8     -4128.1     -4086.5  train
8           3069      3110      13     612      22.2     41.5     -407

In [46]:
# Ls coverage per 30-degree bin: use this to choose the Ls values shown on the map
sols["ls_bin"] = (sols.ls_deg // 30 * 30).astype(int)
cov = sols.groupby("ls_bin").agg(
    sols=("sol", "size"), core_sols=("core_sol", "sum"),
    train_sols=("split_role", lambda s: (s == "train").sum()),
    bins=("n_bins", "sum"),
).reindex(range(0, 360, 30), fill_value=0)
cov.index = [f"{i}-{i + 30}" for i in cov.index]
print(cov.to_string())
print("Ls bins with no training sols (any map Ls there is pure extrapolation):", list(cov.index[cov.train_sols == 0]))

         sols  core_sols  train_sols  bins
0-30       30         26          22  1503
30-60      28         26          23  1394
60-90      17         11          17   760
90-120     23         15          23   999
120-150    23         21          23  1162
150-180    62         38          50  2466
180-210    37         35          37  2035
210-240    19         16          19   948
240-270    14         12          14   669
270-300    20         19          20  1036
300-330    28         27          28  1561
330-360    18         17          18   990
Ls bins with no training sols (any map Ls there is pure extrapolation): []


## Step 2: rover-pixel covariates and the training table

In [47]:
# rover pixel: lon -> col, lat -> row (never swapped)
sols["col"] = np.floor((sols.lon - T_C) / T_A).astype(int)
sols["row"] = np.floor((sols.lat - T_F) / T_E).astype(int)
assert sols.row.between(0, GRID_ROWS - 1).all() and sols.col.between(0, GRID_COLS - 1).all()
sols["pixel_id"] = sols.row * GRID_COLS + sols.col

# consecutive blocks that share a rover pixel are one campaign split by the gap rule, which leaks between CV folds
pixels_by_block = sols.groupby("block").pixel_id.agg(set)
shared = [(b, b + 1) for b in pixels_by_block.index[:-1] if pixels_by_block[b] & pixels_by_block[b + 1]]
assert not shared, f"consecutive blocks share a rover pixel, raise BLOCK_GAP_SOLS to merge them: {shared}"

# known answers from the handover brief
px = sols.set_index("sol")
assert (px.loc[800, "row"], px.loc[800, "col"], px.loc[800, "pixel_id"]) == (304, 372, 449380)
assert (px.loc[1163, "row"], px.loc[1163, "col"], px.loc[1163, "pixel_id"]) == (312, 365, 461189)

needed = set(sols.pixel_id)
use = ["pixel_id", "row", "col", "elev_m", "slope_deg", "aspect_deg", "ti_themis_tiu", "albedo_mean"]
static = pd.concat([c[c.pixel_id.isin(needed)] for c in pd.read_csv(FEATURES_CSV, usecols=use, chunksize=500_000)])
assert static.pixel_id.is_unique and set(static.pixel_id) == needed

# the features CSV's own row/col must agree with ours
chk = sols[["pixel_id", "row", "col"]].drop_duplicates().merge(static[["pixel_id", "row", "col"]], on="pixel_id", suffixes=("", "_csv"))
assert (chk.row == chk.row_csv).all() and (chk.col == chk.col_csv).all()

static = static.drop(columns=["row", "col"]).rename(columns={"elev_m": "elev_dem_m"})
sols = sols.merge(static, on="pixel_id", validate="m:1")
cov_cols = ["elev_dem_m", "slope_deg", "aspect_deg", "ti_themis_tiu", "albedo_mean"]
assert sols[cov_cols].notna().all().all()

# DEM vs REMS elevation; a swapped row/col puts the pixel ~12 km away and breaks this
sols["elev_diff_m"] = sols.elev_dem_m - sols.elev_rems_m
print(sols.elev_diff_m.describe().round(1).to_string())
assert sols.elev_diff_m.abs().median() < 100

count    319.0
mean      15.4
std       15.1
min      -11.7
25%        2.6
50%       17.4
75%       22.9
max       66.8


In [48]:
S0_1AU, MARS_A_AU, MARS_E, LS_PERI, OBLIQ = 1361.0, 1.5237, 0.0934, 251.0, np.deg2rad(25.19)

def insolation(ls, ltst_hr, lat_deg, slope_deg, aspect_deg, tau):
    """Direct-beam flux (W/m2) on a flat and on a tilted surface, plus sun elevation (deg).
    ltst_hr is local TRUE solar time; aspect = downslope azimuth, degrees clockwise from north. Diffuse light is not included."""
    ls_r = np.deg2rad(ls)
    r_au = MARS_A_AU * (1 - MARS_E ** 2) / (1 + MARS_E * np.cos(ls_r - np.deg2rad(LS_PERI)))
    s_toa = S0_1AU / r_au ** 2
    dec = np.arcsin(np.sin(OBLIQ) * np.sin(ls_r))
    ha = np.deg2rad((ltst_hr - 12.0) * 15.0)
    phi = np.deg2rad(lat_deg)
    east = -np.cos(dec) * np.sin(ha)
    north = np.sin(dec) * np.cos(phi) - np.cos(dec) * np.sin(phi) * np.cos(ha)
    up = np.sin(dec) * np.sin(phi) + np.cos(dec) * np.cos(phi) * np.cos(ha)
    sl, az = np.deg2rad(slope_deg), np.deg2rad(aspect_deg)
    cos_inc = east * np.sin(sl) * np.sin(az) + north * np.sin(sl) * np.cos(az) + up * np.cos(sl)
    atten = np.exp(-tau / np.clip(up, 0.02, None))
    flat = np.where(up > 0, s_toa * up * atten, 0.0)
    tilted = np.where(up > 0, s_toa * np.clip(cos_inc, 0, None) * atten, 0.0)
    return flat, tilted, np.rad2deg(np.arcsin(np.clip(up, -1, 1)))

# known answers from the handover brief: flat-surface noon flux at tau 0.5 was ~410 W/m2 (sol 800) and ~247 W/m2 (sol 1163)
for sol, expected in [(800, 410.0), (1163, 247.0)]:
    r = px.loc[sol]
    flux = float(insolation(ls_deg(sol), 12.0, r.lat, 0.0, 0.0, TAU_PLACEHOLDER)[0])
    print(f"sol {sol}: flat noon flux {flux:.1f} W/m2 (brief: ~{expected:.0f})")
    assert abs(flux / expected - 1) < 0.03

sol 800: flat noon flux 408.3 W/m2 (brief: ~410)
sol 1163: flat noon flux 246.0 W/m2 (brief: ~247)


## Step 3: Mastcam dust opacity (tau) per sol

In [49]:
def read_tau(path):
    lines = Path(path).read_text().splitlines()
    start = next(i for i, l in enumerate(lines) if l.strip().startswith("Product_ID"))
    d = pd.read_csv(path, skiprows=start + 1, names=["product_id", "sol_frac", "ls", "tau", "sigma"],
                    skipinitialspace=True, na_values=["******"])
    assert (d.tau > 0).all()                      # -1 would mark an image without validated data
    assert not d.product_id.duplicated().any()
    d["sol"] = np.floor(d.sol_frac).astype(int)   # fractional part of sol_frac is LTST / 24
    return d

raw = {w: read_tau(p) for w, p in TAU_FILES.items()}
by_wl = pd.concat({w: d.groupby("sol").tau.mean() for w, d in raw.items()}, axis=1)
ratio = (by_wl[440] / by_wl[880]).median()        # tau is nearly gray: 440 nm is a few percent below 880 nm
print(f"median tau440 / tau880 = {ratio:.3f};", "sols with only one wavelength:", int(by_wl.isna().any(axis=1).sum()))
by_wl[440] = by_wl[440].fillna(by_wl[880] * ratio)
by_wl[880] = by_wl[880].fillna(by_wl[440] / ratio)
meas = by_wl.mean(axis=1)                           # one tau per measured sol (about the middle of the visible)
meas_n = pd.concat({w: d.groupby("sol").size() for w, d in raw.items()}, axis=1).max(axis=1)
print("measured sols:", len(meas), "sol", meas.index.min(), "to", meas.index.max())

m_sol, m_tau = meas.index.to_numpy(), meas.to_numpy()
s = sols.sol.to_numpy()
j = np.searchsorted(m_sol, s)                       # first measured sol >= s
has_hi, has_lo = j < len(m_sol), j > 0
same = has_hi & (m_sol[np.minimum(j, len(m_sol) - 1)] == s)
lo = np.where(has_lo, m_sol[np.maximum(j - 1, 0)], -10**6)
hi = np.where(has_hi, m_sol[np.minimum(j, len(m_sol) - 1)], 10**6)
interp = ~same & has_lo & has_hi & (hi - lo <= TAU_MAX_GAP_SOLS)

sols["tau"] = TAU_PLACEHOLDER
sols["tau_source"] = np.where(same, "same_sol", np.where(interp, "interpolated", "placeholder"))
sols.loc[same | interp, "tau"] = np.interp(s[same | interp], m_sol, m_tau)
sols["tau_gap_sols"] = np.minimum(s - lo, hi - s)   # distance to the nearest measured sol; 0 on a measured sol
sols["tau_n_img"] = meas_n.reindex(sols.sol).fillna(0).astype(int).to_numpy()
assert (sols.tau > 0).all()
assert (sols.loc[sols.tau_source == "placeholder", "tau"] == TAU_PLACEHOLDER).all()

print(sols.tau_source.value_counts().to_string())
print("sols on the placeholder:", sols.loc[sols.tau_source == "placeholder", "sol"].tolist())
print(sols.groupby("block").agg(sols=("sol", "size"), tau_min=("tau", "min"), tau_median=("tau", "median"), tau_max=("tau", "max"),
                                 placeholder=("tau_source", lambda x: (x == "placeholder").sum())).round(2).to_string())

# how good is the interpolation? hide each measured sol in turn and predict it from its two measured neighbours
a, b, c = m_sol[:-2], m_sol[1:-1], m_sol[2:]
pred = m_tau[:-2] + (m_tau[2:] - m_tau[:-2]) * (b - a) / (c - a)
loo = pd.DataFrame({"gap": c - a, "abs_rel_err": np.abs(pred - m_tau[1:-1]) / m_tau[1:-1]})
loo["gap_bin"] = pd.cut(loo.gap, [0, 4, 10, 20, 30, 10**4])
print(loo.groupby("gap_bin", observed=True).abs_rel_err.agg(n="size", median="median", p90=lambda x: x.quantile(0.9)).round(3).to_string())

median tau440 / tau880 = 0.946; sols with only one wavelength: 59
measured sols: 1360 sol 33 to 3953
tau_source
interpolated    208
same_sol         82
placeholder      29
sols on the placeholder: [1, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 375, 1770, 1775, 1780]
       sols  tau_min  tau_median  tau_max  placeholder
block                                                 
0        90     0.50        0.69     1.16           25
1        11     0.50        0.77     0.78            1
2        53     0.36        0.74     1.47            0
3         6     0.51        0.52     0.53            0
4        12     0.65        0.74     0.78            0
5        40     0.42        0.64     1.09            3
6        24     0.36        0.47     0.74            0
7        23     0.75        0.89     1.14            0
8        13     0.55        0.62     0.65            0
9        34     0.68        1.11     1.68            0
10       13     0.56 

In [50]:
carry = ["sol", "ls_deg", "n_bins", "core_sol", "block", "split_role", "loc_this_sol", "row", "col", "pixel_id",
         "elev_dem_m", "slope_deg", "aspect_deg", "ti_themis_tiu", "albedo_mean", "elev_diff_m",
         "tau", "tau_source", "tau_gap_sols", "tau_n_img"]
t = rems.drop(columns="hour").merge(sols[carry].rename(columns={"n_bins": "sol_n_bins"}), on="sol", validate="m:1")
assert len(t) == len(rems)

t["ground_missing"] = t.ground_temp_k.isna()
t["air_missing"] = t.air_temp_k.isna()

hour_angle = 2 * np.pi * t.ltst_bin_center_hr / 24
ls_angle = np.deg2rad(t.ls_deg)
t["sin_ltst"], t["cos_ltst"] = np.sin(hour_angle), np.cos(hour_angle)
t["sin_ls"], t["cos_ls"] = np.sin(ls_angle), np.cos(ls_angle)
t["sin_aspect"], t["cos_aspect"] = np.sin(np.deg2rad(t.aspect_deg)), np.cos(np.deg2rad(t.aspect_deg))

t["tau_is_placeholder"] = t.tau_source.eq("placeholder")
t["flux_flat_wm2"], t["flux_tilted_wm2"], t["sun_elev_deg"] = insolation(
    t.ls_deg, t.ltst_bin_center_hr, t.latitude_deg, t.slope_deg, t.aspect_deg, t.tau)

print(t.shape)
print("bins missing ground temp:", int(t.ground_missing.sum()), f"({t.ground_missing.mean():.1%});",
      "missing air temp:", int(t.air_missing.sum()))
print(t.split_role.value_counts().to_string())
print(t.groupby("split_role").agg(sols=("sol", "nunique"), rows=("sol", "size")))
print(t[["ltst_bin_center_hr", "ground_temp_k", "flux_flat_wm2", "flux_tilted_wm2"]].describe().round(1).to_string())

(15523, 50)
bins missing ground temp: 664 (4.3%); missing air temp: 96
split_role
train    14350
test      1173
            sols   rows
split_role             
test          25   1173
train        294  14350
       ltst_bin_center_hr  ground_temp_k  flux_flat_wm2  flux_tilted_wm2
count             15523.0        14859.0        15523.0          15523.0
mean                 11.9          228.1           71.4             71.8
std                   6.8           29.5          101.9            102.6
min                   0.1          165.9            0.0              0.0
25%                   6.1          202.1            0.0              0.0
50%                  11.9          219.8            0.0              0.0
75%                  17.6          256.5          145.0            145.8
max                  23.9          287.0          358.6            361.8


In [51]:
OUT_DIR.mkdir(exist_ok=True)
t.to_csv(OUT_DIR / "gale_rems_training.csv", index=False)
sols.drop(columns="ls_bin").to_csv(OUT_DIR / "rems_sol_profile.csv", index=False)
blocks.to_csv(OUT_DIR / "rems_blocks.csv")
print("wrote gale_rems_training.csv, rems_sol_profile.csv, rems_blocks.csv to", OUT_DIR)

wrote gale_rems_training.csv, rems_sol_profile.csv, rems_blocks.csv to processed_data


## Why not random rows (optional check)

Same model and features, three ways of splitting the training-role rows (the sealed test blocks stay untouched). Random rows put 15-minute neighbours of every test row into training, so the score mostly measures interpolation inside sols the model has already seen. The deployment question is a place and season the model has not seen.

In [52]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold, GroupKFold, LeaveOneGroupOut

d = t[(t.split_role == "train") & ~t.ground_missing].reset_index(drop=True)
feats = ["sin_ltst", "cos_ltst", "sin_ls", "cos_ls", "elev_dem_m"]
X, y = d[feats], d.ground_temp_k.to_numpy()

def cv_errors(splitter, groups=None):
    err = np.full(len(d), np.nan)
    for tr, te in splitter.split(X, y, groups):
        m = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.08, random_state=0).fit(X.iloc[tr], y[tr])
        err[te] = m.predict(X.iloc[te]) - y[te]
    return err

results = {
    "random rows (5-fold)": cv_errors(KFold(5, shuffle=True, random_state=0)),
    "whole sols (5-fold)": cv_errors(GroupKFold(5), d.sol.to_numpy()),
    "whole blocks (leave one out)": cv_errors(LeaveOneGroupOut(), d.block.to_numpy()),
}
print(f"target std: {y.std():.1f} K")
for name, e in results.items():
    print(f"{name:30s} RMSE {np.sqrt(np.mean(e ** 2)):5.2f} K   bias {e.mean():+5.2f} K")

e = results["whole blocks (leave one out)"]
per_block = pd.DataFrame({"block": d.block, "e2": e ** 2}).groupby("block").e2.agg(["size", "mean"])
per_block["rmse_K"] = np.sqrt(per_block["mean"])
print(per_block[["size", "rmse_K"]].round(2).to_string())

target std: 29.5 K
random rows (5-fold)           RMSE  2.48 K   bias -0.01 K
whole sols (5-fold)            RMSE  3.32 K   bias -0.11 K
whole blocks (leave one out)   RMSE  5.27 K   bias +0.43 K
       size  rmse_K
block              
0      3997    5.04
1       585    6.18
2      2487    5.01
3       233    3.95
5      1714    4.59
6      1105    6.86
7      1199    4.42
9      1824    5.87
10      582    5.34
